In [6]:
import h5py
from pathlib import Path
import numpy as np
import gvar as gv
import lsqfit

"""Reads twopt_source_averaged.h5 which has all the source-averaged 2pts"""
"""Shape of input 2pt: [n_cfg,n_mom,tsink]"""

PT2_PATH = Path("./twopt_source_averaged.h5")
FIT_PATH = Path("./twopt_fit_results.h5")
Gs, Gt = 32, 96

MOM_LIST = [(0, 0, pz) for pz in range(7)]
TMIN, TMAX = 3, 18
TSEP_LIST = [4, 5, 6, 7]  #(c1*e^-E1tsep) / (c0*e^-E0tsep) will be printed out
N_STATES = 3
SVDCUT = 1e-12

MASS = 0.141
E0_WIDTH = 0.10                  # default width, used when mom is not in E0_PRIOR
E0_PRIOR = {
    (0, 0, 5): gv.gvar(0.9561, 0.2), #Take prior from the mean of lattice dispersion relation and continuous dispersion relation
    (0, 0, 6): gv.gvar(1.1275, 0.2), #Give wide prior for pz=5,6
}

DE_PRIOR = {
}

DE1_PRIOR = gv.gvar("0.60(0.40)")
DE2_PRIOR = gv.gvar("0.60(0.40)")
C_CENTER = 0.0
C_WIDTH = 1e4

PROGRESS_EVERY = 100

tag = (f"nstate{N_STATES}_t{TMIN}-{TMAX}_svd{SVDCUT:.0e}" + ("_e0prior" if E0_PRIOR else "") + ("_deprior" if DE_PRIOR else ""))


def twopt_model(t, p):
    dE = np.exp(p["log(dE)"])      # exponentiate the fit parameter explicitly
    corr = 0.0
    e = 0.0
    for n in range(len(dE)):
        e = e + dE[n]
        corr = corr + p["c"][n] * (gv.exp(-e * t) + gv.exp(-e * (Gt - t)))
    return corr


def make_prior(n_states, mom):
    prior = gv.BufferDict()

    if mom in E0_PRIOR:
        e0, e0_width = gv.mean(E0_PRIOR[mom]), gv.sdev(E0_PRIOR[mom])
    else:
        p_lat = 2.0 * np.pi * np.array(mom, dtype=float) / Gs
        e0 = 2.0 * np.arcsinh(np.sqrt(np.sinh(MASS / 2.0) ** 2
                                      + np.sum(np.sin(p_lat / 2.0) ** 2)))
        e0_width = E0_WIDTH

    centers = [e0]
    widths = [e0_width]

    if n_states == 2:
        # one gap: dE1
        if (1, mom) in DE_PRIOR:
            gap1 = DE_PRIOR[(1, mom)]
        else:
            gap1 = DE1_PRIOR
        centers.append(gv.mean(gap1))
        widths.append(gv.sdev(gap1))

    elif n_states == 3:
        # two gaps: dE1 and dE2
        if (1, mom) in DE_PRIOR:
            gap1 = DE_PRIOR[(1, mom)]
        else:
            gap1 = DE1_PRIOR
        centers.append(gv.mean(gap1))
        widths.append(gv.sdev(gap1))

        if (2, mom) in DE_PRIOR:
            gap2 = DE_PRIOR[(2, mom)]
        else:
            gap2 = DE2_PRIOR
        centers.append(gv.mean(gap2))
        widths.append(gv.sdev(gap2))

    elif n_states != 1:
        raise ValueError(f"make_prior only handles n_states = 1, 2, 3, got {n_states}")

    prior["c"] = gv.gvar([C_CENTER] * n_states, [C_WIDTH] * n_states)
    prior["log(dE)"] = gv.log(gv.gvar(centers, widths))
    return prior


def cov_of_mean(samples):
    return np.cov(samples, rowvar=False, ddof=1) / samples.shape[0]


def jk_stats(values):
    """Jackknife mean and error over axis 0 of delete-one samples."""
    n = values.shape[0]
    mean = values.mean(axis=0)
    return mean, np.sqrt((n - 1.0) / n * np.sum((values - mean) ** 2, axis=0))


with h5py.File(PT2_PATH, "r") as f:
    name = "correlator_cfg" if "correlator_cfg" in f else "pion_45"
    pion = f[name][:]
    if "momentum_list" in f:
        moms = f["momentum_list"][:]
    else:
        moms = f[name].attrs["momentums"]

mom_to_idx = {tuple(p): i for i, p in enumerate(moms.tolist())}
t = np.arange(TMIN, TMAX + 1)
n_jk = pion.shape[0]


for pf in MOM_LIST:
    ipf = mom_to_idx[pf]
    C2_pf = pion[..., ipf, :].real
    samples = C2_pf[:, TMIN:TMAX + 1]
    mean = samples.mean(axis=0)
    cov = cov_of_mean(samples)
    jk = (samples.sum(axis=0) - samples) / (n_jk - 1)

    prior = make_prior(N_STATES, pf)

    central_fit = lsqfit.nonlinear_fit(data=(t, gv.gvar(mean, cov)),
                                       fcn=twopt_model, prior=prior,
                                       svdcut=SVDCUT)
    E = np.cumsum(central_fit.p["dE"])  #[E0, dE1, dE2] -> [E0, E1, E2]
    E_central = gv.mean(E)
    E_central_sdev = gv.sdev(E)
    C_central = gv.mean(central_fit.p["c"])
    C_central_sdev = gv.sdev(central_fit.p["c"])
    chi2dof_central = central_fit.chi2 / central_fit.dof
    Q_central = central_fit.Q

    print(f"\n====== CENTRAL FIT RESULT   pf = {pf} ======")
    print(f"  central: n_states = {N_STATES}   t = {TMIN}..{TMAX}")
    for n in range(N_STATES):
        print(f"  central: E{n} = {E[n]:<14} c{n} = {central_fit.p['c'][n]}")
    print(f"  central: chi2/dof = {chi2dof_central:.2f}   "
          f"Q = {Q_central:.3f}   svdn = {getattr(central_fit, 'svdn', 0)}")
    print(central_fit.format(maxline=True))

    if N_STATES >= 2:
        ratio = (central_fit.p["c"][1] / central_fit.p["c"][0]
             * gv.exp(-central_fit.p["dE"][1] * np.array(TSEP_LIST)))
        for tsep, r in zip(TSEP_LIST, ratio):
            print(f"  central: C1 e^(-E1 tsep) / C0 e^(-E0 tsep) "
              f"at tsep = {tsep}: {r}")

    #per-jackknife central values
    E_jk = np.zeros((n_jk, N_STATES))
    dE_jk = np.zeros((n_jk, N_STATES))
    C_jk = np.zeros((n_jk, N_STATES))

    #per-jackknife sdev: each replicate's own fit error, not the scatter across them
    E_sdev_jk = np.zeros((n_jk, N_STATES))
    dE_sdev_jk = np.zeros((n_jk, N_STATES))
    C_sdev_jk = np.zeros((n_jk, N_STATES))

    #per-jackknife fit quality
    chi2dof_jk = np.zeros(n_jk)
    Q_jk = np.zeros(n_jk)

    p0 = {k: central_fit.pmean[k] for k in prior}
    for i in range(n_jk):
        fit_jk = lsqfit.nonlinear_fit(data=(t, gv.gvar(jk[i], cov)),
                                      fcn=twopt_model, prior=prior,
                                      svdcut=SVDCUT, p0=p0)
        #save central values
        dE_jk[i] = gv.mean(fit_jk.p["dE"])
        E_jk[i] = np.cumsum(dE_jk[i])
        C_jk[i] = gv.mean(fit_jk.p["c"])

        #save sdevs (cumsum the gvars so E keeps the dE correlations)
        E_sdev_jk[i]  = gv.sdev(np.cumsum(fit_jk.p["dE"]))
        dE_sdev_jk[i] = gv.sdev(fit_jk.p["dE"])
        C_sdev_jk[i]  = gv.sdev(fit_jk.p["c"])

        chi2dof_jk[i] = fit_jk.chi2 / fit_jk.dof
        Q_jk[i] = fit_jk.Q
        if PROGRESS_EVERY and (i + 1) % PROGRESS_EVERY == 0:
            print(f"  jackknife {i + 1}/{n_jk}", flush=True)

    dE_jk_mean, dE_jk_err = jk_stats(dE_jk)
    E_jk_mean, E_jk_err = jk_stats(E_jk)
    C_jk_mean, C_jk_err = jk_stats(C_jk)
    chi2_jk_mean, chi2_jk_err = jk_stats(chi2dof_jk)
    Q_jk_mean, Q_jk_err = jk_stats(Q_jk)

    #here construct second / first again using jackknife results
    # excited-state contamination measured as (C1/C0) e^{-dE1 tsep}, built per-jk
    if N_STATES >= 2:
        tsep_arr = np.array(TSEP_LIST, dtype=float)
        contam_jk = ((C_jk[:, 1] / C_jk[:, 0])[:, None]
                     * np.exp(-dE_jk[:, 1][:, None] * tsep_arr[None, :]))
        contam_mean, contam_err = jk_stats(contam_jk)   # each shape (len(TSEP_LIST),)
    
    print(f"\n====== JACKKNIFE RESULT   pf = {pf} ======")
    print(f"  E0 : jk_mean = {dE_jk_mean[0]:.6f}   jk_err = {dE_jk_err[0]:.6f}")
    for n in range(1, N_STATES):
        print(f"  dE{n}: jk_mean = {dE_jk_mean[n]:.6f}   "
              f"jk_err = {dE_jk_err[n]:.6f}")
    for n in range(N_STATES):
        print(f"  c{n} : jk_mean = {C_jk_mean[n]:.6e}   "
              f"jk_err = {C_jk_err[n]:.6e}")
    for itsep,tsep in enumerate(TSEP_LIST) : 
        print(f"  At tsep={tsep}, 2nd / 1st term is {contam_mean[itsep]:.5f}+-{contam_err[itsep]:.5f}")
    print(f"  chi2/dof: jk_mean = {chi2_jk_mean:.3f}   "
          f"jk_err = {chi2_jk_err:.3f}")
    print(f"  Q       : jk_mean = {Q_jk_mean:.3f}   jk_err = {Q_jk_err:.3f}")
    print(f"\n===========================================")
    with h5py.File(FIT_PATH, "a") as f:
        g = f.require_group(tag)  #get a group by fit parameter
        g.attrs["tmin"] = TMIN
        g.attrs["tmax"] = TMAX
        g.attrs["n_states"] = N_STATES
        g.attrs["svdcut"] = SVDCUT
        g.attrs["n_jk"] = n_jk
        mom_name = f"p{pf[0]}_{pf[1]}_{pf[2]}"
        if mom_name in g:
            del g[mom_name]
        gm = g.create_group(mom_name)  #get a subgroup by momentum
        gm.create_dataset("momentum", data=np.array(pf, dtype=np.int64))
        gm.create_dataset("E_central", data=E_central)
        gm.create_dataset("E_central_sdev", data=E_central_sdev)
        gm.create_dataset("C_central", data=C_central)
        gm.create_dataset("C_central_sdev", data=C_central_sdev)
        gm.create_dataset("chi2dof_central", data=chi2dof_central)
        gm.create_dataset("Q_central", data=Q_central)
        gm.create_dataset("E_jk", data=E_jk)
        gm.create_dataset("dE_jk", data=dE_jk)
        gm.create_dataset("C_jk", data=C_jk)
        gm.create_dataset("E_sdev_jk", data=E_sdev_jk)
        gm.create_dataset("dE_sdev_jk", data=dE_sdev_jk)
        gm.create_dataset("C_sdev_jk", data=C_sdev_jk)
        gm.create_dataset("chi2dof_jk", data=chi2dof_jk)
        gm.create_dataset("Q_jk", data=Q_jk)
        gm.create_dataset("E_jk_mean", data=E_jk_mean)
        gm.create_dataset("E_jk_err", data=E_jk_err)
        gm.create_dataset("dE_jk_mean", data=dE_jk_mean)
        gm.create_dataset("dE_jk_err", data=dE_jk_err)
        gm.attrs["dim_E_jk"] = "jk,state"
        gm.attrs["dim_dE_jk"] = "jk,state (slot 0 is E0, slots 1+ are gaps)"
        gm.attrs["dim_dE_sdev_jk"] = ("jk,state -- lsqfit's uncertainty on each "
                                      "replicate's own fit, NOT the jackknife "
                                      "scatter of dE_jk across replicates")
        g.attrs["e0_prior"] = str({k: str(v) for k, v in E0_PRIOR.items()}) if E0_PRIOR else "none"
        g.attrs["de_prior"] = str({k: str(v) for k, v in DE_PRIOR.items()}) if DE_PRIOR else "none"
    print(f"  written: {FIT_PATH}:{tag}/{mom_name}", flush=True)

print("\nfinished", flush=True)


====== CENTRAL FIT RESULT   pf = (0, 0, 0) ======
  central: n_states = 3   t = 3..18
  central: E0 = 0.14092(17)    c0 = -6.634(15)e-07
  central: E1 = 0.78(25)       c1 = -1.7(2.4)e-07
  central: E2 = 1.58(45)       c2 = -1.64(93)e-06
  central: chi2/dof = 0.92   Q = 0.547   svdn = 0
Least Squares Fit:
  chi2/dof [dof] = 0.92 [16]    Q = 0.55    logGBF = 260.05

Parameters:
            c 0   -6.634(15)e-07       [ 0 ± 1.0e+04 ]  
              1    -1.7(2.4)e-07       [ 0 ± 1.0e+04 ]  
              2    -1.64(93)e-06       [ 0 ± 1.0e+04 ]  
      log(dE) 0     -1.9596 (12)       [  -1.96 (71) ]  
              1       -0.45 (39)       [  -0.51 (67) ]  
              2       -0.22 (28)       [  -0.51 (67) ]  
------------------------------------------------------
           dE 0     0.14092 (17)       [   0.14 (10) ]  
              1        0.64 (25)       [   0.60 (40) ]  
              2        0.81 (23)       [   0.60 (40) ]  

Fit:
     x[k]               y[k]          f(x[k],p